# Micro Proyecto 3 — Finetuning de Llama-3.2-1B con LoRA en RACE

**Maestría en Inteligencia Artificial — Universidad de los Andes**  
**Integrantes — Daniel Gallo - Sebastian Garcia - Nicolaz Gomez**  
*Modelos Avanzados para el Procesamiento de Lenguaje Natural*

---

## Objetivo

Realizar el **finetuning** del modelo `Llama-3.2-1B` mediante **LoRA (Low-Rank Adaptation)** sobre el dataset **RACE** (Reading Comprehension from Examinations) para resolver una tarea de **preguntas de selección múltiple** (4 opciones: A, B, C, D), y comparar su desempeño contra el modelo base sin ajustar.

## Métrica

Accuracy sobre el split de *test*, calculado como:

$$\hat{s}=\text{argmax}_{s\in\left\{ A,B,C,D\right\} }\log\,P\left(s\mid c\right)$$

donde $\hat{s}$ es la letra predicha y $s^{*}$ la respuesta verdadera

$$\text{accuracy}=\cfrac{1}{N}\sum_{i=1}^{N}\left\{ \begin{array}{ccc}
1 & \text{si} & \hat{s}^{(i)}=s^{*(i)}\\
0 & \text{si} & \text{no}
\end{array}\right\} $$

donde $c$ es el contexto (prompt + pregunta) y $P(s\mid c)$ es la probabilidad asignada por el modelo al token de la letra dada el contexto.

## Restricciones del enunciado

- Solo se usan ejemplos con `len(article) < 800` caracteres (en los tres splits).
- El modelo base es `meta-llama/Llama-3.2-1B`.
- Durante el entrenamiento, los tokens del contexto deben tener `label = -100` para ser **ignorados por la función de costo** (el modelo solo aprende a predecir la respuesta).
- Se debe entregar el checkpoint del mejor modelo (vía `torch.save`).


### Actualización de dependencias

Se actualiza `torchao` a la versión `0.16.0` para garantizar compatibilidad con las versiones de `transformers` y `peft` que se encuentran en el runtime de Google Colab.


In [ ]:
! pip install -U torchao==0.16.0 -q

### Credenciales y rutas

Se obtienen el **token de Hugging Face** y el **`SAVE_PATH`** desde los *secrets* de Colab. Si Colab no está disponible (ejecución local), se intenta leerlos como **variables de entorno**. Como último recurso, pueden hardcodearse reemplazando la cadena `"Token"` por el valor correspondiente.

El token es necesario porque `meta-llama/Llama-3.2-1B` es un modelo *gated* en Hugging Face y requiere autenticación para descargarse.


In [ ]:
import os

try:
    from google.colab import userdata,drive
    drive.mount('/content/drive')
    TOKEN=userdata.get('huggingface_token')
    SAVE_PATH=userdata.get('save_path')
except:
    TOKEN = os.getenv('HUGGINGFACE_TOKEN','Token')
    SAVE_PATH=os.getenv('SAVE_PATH','/tmp/microproyecto_3')

## 1. Setup e imports

Se importan las librerías necesarias (`torch`, `transformers`, `peft`, `datasets`, `tqdm`, `matplotlib`) y se definen los **hiperparámetros** globales del experimento:

- **Modelo**: `meta-llama/Llama-3.2-1B`
- **Batch sizes**: 32 (train y eval)
- **Learning rate**: `2e-4` (típico para finetuning con LoRA)
- **Epochs**: 8 (con early stopping)
- **Mixed precision**: `bf16` si el hardware lo soporta, `fp16` en caso contrario
- **LoRA**: `r=32`, `alpha=64`, `dropout=0.2`, target = atención + MLP (ver Sección 9)

También se activan optimizaciones de GPU (TF32, cuDNN benchmark, `matmul precision='high'`) y se fija la semilla para reproducibilidad.


In [ ]:

import math
import pandas as pd
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, get_peft_model_state_dict
from datasets import load_dataset

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from huggingface_hub import login


In [ ]:
# ---------------- Reproducibilidad ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

SAVE_PATH=os.path.join(SAVE_PATH,"microproyecto_3")
os.makedirs(SAVE_PATH, exist_ok=True)

# ---------------- Hiperparámetros ----------------
MODEL_NAME    = "meta-llama/Llama-3.2-1B"
MAX_LENGTH    = 768          # holgura sobre lo que produce article < 800 chars
BATCH_TRAIN   = 32
BATCH_EVAL    = 32
LR            = 2e-4
EPOCHS        = 8
WARMUP_RATIO  = 0.05
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.01

# ---------------- GPU optimizations ----------------
# cudnn.benchmark autotunea kernels; útil siempre que los shapes sean estables
torch.backends.cudnn.benchmark = True
# TF32 solo aplica a operaciones fp32; en bf16/fp16 no tiene efecto.
# Lo dejamos activado por si alguna op cae a fp32 (ej. capas LayerNorm en algunos casos).
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# ---------------- Mixed precision ----------------
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mixed precision: {'bf16' if USE_BF16 else 'fp16'} (AMP_DTYPE={AMP_DTYPE})")

# LoRA
LORA_R           = 32 # antes 8
LORA_ALPHA       = 64 # antes 16
LORA_DROPOUT     = 0.1
LORA_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] # antes ["q_proj", "k_proj", "v_proj", "o_proj"],

In [ ]:
login(token=TOKEN)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"   # defensivo; collate_fn lo dicta de todos modos

print(f"Vocab size:    {tokenizer.vocab_size}")
print(f"BOS: {tokenizer.bos_token_id} ({tokenizer.bos_token!r})")
print(f"EOS: {tokenizer.eos_token_id} ({tokenizer.eos_token!r})")
print(f"PAD: {tokenizer.pad_token_id} ({tokenizer.pad_token!r})")


## 2. Dataset RACE

[**RACE**](https://huggingface.co/datasets/ehovy/race) (*ReAding Comprehension from Examinations*) contiene aproximadamente **100k preguntas** de selección múltiple extraídas de exámenes de inglés en escuelas chinas. Cada ejemplo consta de:

- Un **artículo** (`article`)
- Una **pregunta** (`question`)
- **Cuatro opciones** (`options`: lista de 4 strings)
- La **letra correcta** (`answer` ∈ {A, B, C, D})

### Filtro requerido por el enunciado

Se conservan **únicamente** los ejemplos cuyo artículo tenga **menos de 800 caracteres** (`len(article) < 800`). Este filtro se aplica a los **tres splits** (`train`, `validation`, `test`):

- Reduce el tiempo de entrenamiento sin alterar la naturaleza de la tarea.
- Hace que las secuencias quepan cómodamente en `MAX_LENGTH=768` tokens.
- Mantiene la evaluación coherente con el dataset filtrado (no se evalúa sobre ejemplos que el modelo no vio durante el entrenamiento).


In [ ]:
dataset = load_dataset("ehovy/race", "all")

def short_article(example):
    return len(example["article"]) < 800

train_ds_raw = dataset["train"].filter(short_article)
val_ds_raw   = dataset["validation"].filter(short_article)
test_ds_raw  = dataset["test"].filter(short_article)

print(f"train: {len(train_ds_raw):>6}")
print(f"val:   {len(val_ds_raw):>6}")
print(f"test:  {len(test_ds_raw):>6}")

# Inspección rápida de un ejemplo
ex = train_ds_raw[0]
print("\n--- Ejemplo ---")
print(f"Article: {ex['article'][:200]}...")
print(f"Question: {ex['question']}")
print(f"Options: {ex['options']}")
print(f"Answer: {ex['answer']}")


## 3. Construcción del prompt

El diseño del prompt es crítico para tareas con modelos causales. Diseñamos uno que:

1. **Da instrucciones explícitas** al modelo: debe responder con una sola letra (A, B, C o D).
2. **Estructura el contexto** de forma clara y consistente: instrucción → artículo → pregunta → opciones → marca de respuesta.
3. **Termina exactamente en `"Answer:"`** (sin espacio final).

### Estructura del prompt

```
Read the article and answer the multiple-choice question
by selecting the letter (A, B, C, or D) of the correct option.

Article: <article>

Question: <question>

A) <option_a>
B) <option_b>
C) <option_c>
D) <option_d>

Answer:
```

In [ ]:
def build_prompt(example):
    PROMPT_TEMPLATE = (
        "Read the article and answer the multiple-choice question "
        "by selecting the letter (A, B, C, or D) of the correct option.\n\n"
        "Article: {article}\n\n"
        "Question: {question}\n\n"
        "A) {opt_a}\n"
        "B) {opt_b}\n"
        "C) {opt_c}\n"
        "D) {opt_d}\n\n"
        "Answer:"
    )
    return PROMPT_TEMPLATE.format(
        article=example["article"].strip(),
        question=example["question"].strip(),
        opt_a=example["options"][0],
        opt_b=example["options"][1],
        opt_c=example["options"][2],
        opt_d=example["options"][3],
    )


## 4. Tokenización

La tokenización es uno de los puntos más delicados del proyecto. Los tokenizers basados en **BPE (Byte-Pair Encoding)** —como el de Llama— tratan **el espacio como parte del siguiente token**, no como un separador independiente. Esto tiene dos consecuencias críticas:

1. `tokenizer("A")` y `tokenizer(" A")` producen **token IDs distintos**.
2. Si el prompt termina en `"Answer:"` y la respuesta esperada es la letra `A`, el modelo debe predecir el token `" A"` (con espacio inicial), no `"A"`.

Por eso prefijamos la respuesta con un espacio (`answer = " " + example["answer"]`) y, más adelante (Sección 7), verificamos con `assert` que cada uno de `" A"`, `" B"`, `" C"`, `" D"` sea **un único token**. Esta propiedad permite calcular $P(s \mid c)$

### Dos modos de tokenización

Se implementan dos funciones distintas porque entrenamiento y evaluación necesitan estructuras de datos diferentes:

| Aspecto         | `tokenize_for_training`                       | `tokenize_for_testing`                       |
|-----------------|-----------------------------------------------|----------------------------------------------|
| Contenido       | prompt + respuesta + EOS                      | solo prompt                                  |
| `labels`        | `-100` en el contexto, tokens reales en la respuesta | no se generan                          |
| BOS / EOS       | BOS al inicio, EOS al final                   | BOS al inicio, **sin** EOS                   |
| Campo extra     | —                                             | `answer` (letra correcta, para accuracy)     |
| Uso             | cómputo de la *cross-entropy loss*            | extracción de `logits[:, -1, :]` para argmax |

### Por qué `labels = -100` en el contexto

En PyTorch, la `CrossEntropyLoss` **ignora** cualquier posición cuyo label sea `-100`. Esto implementa exactamente el requisito del enunciado:

> *Durante el entrenamiento se debe ignorar los tokens que hacen parte del contexto, con el objetivo de que el modelo aprenda a generar únicamente la respuesta.*

Sin este enmascarado, el modelo sería penalizado también por reproducir el artículo y la pregunta, lo que diluiría la señal de entrenamiento: aprendería a copiar el contexto en lugar de a responderlo.

### Encodeo separado del prompt y de la respuesta

Encodeamos `prompt` y `answer` **por separado** y concatenamos los IDs resultantes. Esto evita un problema sutil de **fronteras BPE**: si concatenáramos los strings y tokenizáramos en un solo paso, el algoritmo BPE podría fusionar caracteres a través de la frontera prompt/respuesta y alterar el token objetivo. Concatenar a nivel de IDs garantiza que `" A"` quede como su token único.

### Por qué no incluir EOS en testing

En testing solo necesitamos hacer un forward del prompt y leer `logits[:, -1, :]`. Añadir EOS al final del prompt cambiaría la posición de la última predicción válida y rompería la evaluación.


In [ ]:
def tokenize_for_training(example, tokenizer=tokenizer):
    prompt = build_prompt(example)
    answer = " " + example["answer"].strip()  # " A" / " B" / " C" / " D"

    prompt_tokens = tokenizer(prompt, add_special_tokens=True)
    answer_tokens = tokenizer(answer, add_special_tokens=False)
    prompt_ids = prompt_tokens["input_ids"]
    answer_ids = answer_tokens["input_ids"]

    input_ids      = prompt_ids + answer_ids + [tokenizer.eos_token_id]
    attention_mask = prompt_tokens["attention_mask"] + answer_tokens["attention_mask"] + [1]
    labels         = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]

    assert len(input_ids) == len(labels) == len(attention_mask)
    assert input_ids[-1] == tokenizer.eos_token_id
    assert input_ids[0]  == tokenizer.bos_token_id
    assert labels[len(prompt_ids)] == answer_ids[0]

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }


In [ ]:
def tokenize_for_testing(example, tokenizer=tokenizer):
    prompt = build_prompt(example)

    prompt_tokens  = tokenizer(prompt, add_special_tokens=True)
    prompt_ids     = prompt_tokens["input_ids"]
    attention_mask = prompt_tokens["attention_mask"]

    assert len(prompt_ids) == len(attention_mask)
    assert prompt_ids[0]  == tokenizer.bos_token_id
    assert prompt_ids[-1] != tokenizer.eos_token_id  # NO debe terminar en EOS

    return {
        "input_ids":      prompt_ids,
        "attention_mask": attention_mask,
        "answer":         example["answer"].strip(),  # se preserva para computar accuracy
    }


### Aplicación de la tokenización a cada split

Se mapea la función de tokenización correspondiente sobre cada split del dataset:

- **`train`** → `tokenize_for_training` (incluye respuesta y `labels`)
- **`val`** y **`test`** → `tokenize_for_testing` (solo prompt, con campo `answer`)
- **`val_loss`** (segunda versión de `val`) → `tokenize_for_training`, usada durante el entrenamiento para monitorear la *cross-entropy* de validación junto al accuracy

Las columnas originales del dataset (`article`, `question`, `options`, ...) se eliminan tras el mapeo porque su contenido ya está codificado en `input_ids`. Esto reduce el uso de memoria y acelera el `DataLoader`.


In [ ]:
# Aplicamos tokenización a cada split.
# - train usa el formato con labels para entrenar
# - val y test usan el formato test-style (solo prompt + answer correcto)

train_ds = train_ds_raw.map(
    tokenize_for_training,
    remove_columns=train_ds_raw.column_names,
    desc="Tokenizing train",
)

val_ds_for_acc = val_ds_raw.map(
    tokenize_for_testing,
    remove_columns=val_ds_raw.column_names,
    desc="Tokenizing val (test-style)",

)

val_ds_for_loss = val_ds_raw.map(
    tokenize_for_training,
    remove_columns=val_ds_raw.column_names,
    desc="Tokenizing val (train-style for loss)",
)

test_ds = test_ds_raw.map(
    tokenize_for_testing,
    remove_columns=test_ds_raw.column_names,
    desc="Tokenizing test",
)

# Inspección de longitudes para verificar que MAX_LENGTH es suficiente
lengths = [len(x["input_ids"]) for x in train_ds]
print(f"Train token lengths — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.0f}, p99: {np.percentile(lengths, 99):.0f}")
print(f"MAX_LENGTH = {MAX_LENGTH} (debería ser >= max)")


## 5. Collate functions y padding asimétrico

El *collate function* se invoca por cada batch y aplica **padding dinámico**: cada batch se rellena hasta la longitud del ejemplo más largo del propio batch (no hasta `MAX_LENGTH` global). Adicionalmente, esa longitud se redondea al **múltiplo de 8 más cercano** para activar los **Tensor Cores** en `bf16`/`fp16`.

### Por qué padding distinto en training vs. testing

|                   | Training (`collate_fn_train`)     | Testing (`collate_fn_test`)        |
|-------------------|-----------------------------------|------------------------------------|
| Lado de padding   | **derecha**                       | **izquierda**                      |
| Padding en labels | `-100` (ignorado por la loss)     | no aplica                          |

**Razón en training (right padding):**  
El modelo causal hace *shift interno* de los labels durante el cómputo de la loss. Con padding a la derecha, la respuesta queda alineada justo después del prompt y los tokens de pad terminan al final, donde sus labels son `-100` y no contaminan el cómputo de la cross-entropy.

**Razón en testing (left padding):**  
Para evaluación leemos `logits[:, -1, :]` — la última posición de cada secuencia. Con **left padding**, el último token real del prompt cae exactamente en la columna `-1` para **todo** el batch, sin importar las diferencias de longitud entre ejemplos. Con right padding, el último token real de los ejemplos cortos quedaría rodeado de pads y `logits[:, -1, :]` apuntaría a una posición de relleno — el accuracy sería incorrecto.

Este detalle es invisible en el resultado final pero **fundamental para que la evaluación batched sea válida**.


In [ ]:
def collate_fn_train(batch, pad_id=tokenizer.pad_token_id, pad_to_multiple_of=8):
    max_len = max(len(x["input_ids"]) for x in batch)
    if pad_to_multiple_of:
        m = pad_to_multiple_of
        max_len = ((max_len + m - 1) // m) * m

    ids, attn, lbl = [], [], []
    for x in batch:
        n   = len(x["input_ids"])
        pad = max_len - n
        # RIGHT padding
        ids .append(x["input_ids"]      + [pad_id] * pad)
        attn.append(x["attention_mask"] + [0]      * pad)
        lbl .append(x["labels"]         + [-100]   * pad)

    return {
        "input_ids":      torch.tensor(ids,  dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels":         torch.tensor(lbl,  dtype=torch.long),
    }


In [ ]:
def collate_fn_test(batch, pad_id=tokenizer.pad_token_id, pad_to_multiple_of=8):
    max_len = max(len(x["input_ids"]) for x in batch)
    if pad_to_multiple_of:
        m = pad_to_multiple_of
        max_len = ((max_len + m - 1) // m) * m

    ids, attn, answers = [], [], []
    for x in batch:
        n   = len(x["input_ids"])
        pad = max_len - n
        # LEFT padding
        ids .append([pad_id] * pad + x["input_ids"])
        attn.append([0]      * pad + x["attention_mask"])
        answers.append(x["answer"])

    return {
        "input_ids":      torch.tensor(ids,  dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "answers":        answers,   # lista de strings, no tensor
    }


## 6. DataLoaders

Se construyen los `DataLoader` de PyTorch para cada split. Detalles:

- **`train_loader`**: `shuffle=True`, `drop_last=True` (descarta el último batch incompleto para mantener todos los batches del mismo tamaño y estabilizar las estadísticas de mixed precision).
- **`val_loader`** y **`test_loader`**: usan `collate_fn_test` (left padding); no se mezclan ejemplos (`shuffle=False`).
- **`val_loader_loss`**: variante de validación con `collate_fn_train` (right padding + labels), necesaria para calcular la *cross-entropy de validación* durante el entrenamiento como señal complementaria al accuracy.
- `pin_memory=True`: acelera la transferencia CPU → GPU en máquinas con CUDA.


In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn_train, num_workers=0,
    pin_memory=True, drop_last=True,
)

val_loader = DataLoader(
    val_ds_for_acc, batch_size=BATCH_EVAL, shuffle=False,
    collate_fn=collate_fn_test, num_workers=0,
    pin_memory=True, drop_last=False,
)

val_loader_loss = DataLoader(
    val_ds_for_loss, batch_size=BATCH_EVAL, shuffle=False,
    collate_fn=collate_fn_train,
    num_workers=0, pin_memory=True, drop_last=False,
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_EVAL, shuffle=False,
    collate_fn=collate_fn_test, num_workers=0,
    pin_memory=True, drop_last=False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


## 7. Utilidades de evaluación

### La métrica del enunciado

El proyecto pide medir el accuracy seleccionando, para cada ejemplo, la opción con mayor probabilidad condicional dado el contexto:

$$
\hat{s}=\text{argmax}_{s\in\left\{ A,B,C,D\right\} }\log\,P\left(s\mid c\right)
$$

donde $c$ es el prompt (instrucción + artículo + pregunta + opciones + `"Answer:"`). Como cada opción se reduce a **un único token** (`" A"`, `" B"`, `" C"`, `" D"`), la expresión se simplifica:

$$
P(s \mid c) = \mathrm{softmax}(\mathrm{logits}_{-1})[s]
$$

Basta entonces con **un solo forward pass** del prompt y leer los logits de la última posición. No es necesario generar texto autoregresivamente ni acumular log-probabilidades sobre varios tokens (lo cual sería obligatorio si las respuestas tuvieran longitud variable, por ejemplo si fueran las propias frases de las opciones en vez de la letra).

### Funciones implementadas

1. **`get_letter_token_ids`** — recupera los token IDs de `" A"`, `" B"`, `" C"`, `" D"` y verifica con `assert` que cada uno sea **un único token**. Esta verificación es crítica: si por algún motivo `" A"` se tokenizara en dos sub-tokens, toda la lógica de evaluación dejaría de ser válida.

2. **`evaluate_accuracy`** — itera el dataloader bajo `torch.no_grad()` y `autocast`, hace el forward del prompt, extrae `logits[:, -1, :]` (válido para todo el batch gracias al **left padding**), aplica `log_softmax` y selecciona el `argmax` **restringido a los 4 token IDs** de las letras. Además calcula:
   - **Distribución de predicciones por letra** → permite detectar sesgos (¿el modelo predice siempre la misma letra?).
   - **Accuracy por letra correcta** → muestra si el modelo es más débil en alguna clase.

3. **`compute_val_loss`** — calcula la cross-entropy promedio sobre el dataloader de validación con labels. Sirve como **segunda señal de overfitting**, complementaria al accuracy: la loss puede comenzar a subir varios epochs antes de que el accuracy se degrade.

4. **`generate_examples`** — genera respuestas con `model.generate()` (decoding greedy) sobre un subconjunto aleatorio del dataset, para **inspección cualitativa**. Permite comparar visualmente si el modelo finetuneado produce respuestas más limpias y directas que el modelo base.

### Por qué argmax sobre 4 tokens y no `generate`

Generar tokens y luego parsear la letra sería:

- **Más lento**: requiere decoding autoregresivo (al menos un token por ejemplo, pero típicamente varios).
- **Más frágil**: el modelo puede emitir `"A) ..."`, `"Answer: A"`, `"The answer is A"`, etc., y parsear todos estos casos requiere reglas ad-hoc que pueden fallar.
- **Menos fiel a la métrica del enunciado**: $\arg\max P(s \mid c)$ es una probabilidad sobre un conjunto fijo de opciones, no una generación libre.

El argmax restringido a los 4 token IDs nos da exactamente la métrica pedida, en una sola pasada y sin ambigüedad.


In [ ]:
def get_letter_token_ids(tokenizer):
    """Devuelve dict letra -> token_id para ' A', ' B', ' C', ' D'."""
    out = {}
    for letter in ["A", "B", "C", "D"]:
        ids = tokenizer.encode(" " + letter, add_special_tokens=False)
        assert len(ids) == 1, f"' {letter}' debería ser 1 token, dio {ids}"
        out[letter] = ids[0]
    return out

# Verificación
LETTER_IDS = get_letter_token_ids(tokenizer)
print("Token IDs de las opciones:")
for l, tid in LETTER_IDS.items():
    print(f"  ' {l}' -> {tid}")


In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, dataloader, device="cuda",
                      max_examples=None, desc="Eval"):
    """Calcula accuracy usando argmax_s log P(s|c). Asume left padding."""
    model.eval()
    letters = ["A", "B", "C", "D"]
    letter_ids = get_letter_token_ids(tokenizer)
    letter_id_t = torch.tensor([letter_ids[l] for l in letters], device=device)

    correct, total = 0, 0
    per_letter_pred = {l: 0 for l in letters}
    per_letter_true = {l: {"correct": 0, "total": 0} for l in letters}

    for batch in tqdm(dataloader, desc=desc):
        if max_examples is not None and total >= max_examples:
            break

        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        true_answers   = batch["answers"]

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # left padding -> última posición real = -1 para todo el batch
        logits     = outputs.logits[:, -1, :]
        log_probs  = torch.log_softmax(logits.float(), dim=-1)
        scores     = log_probs[:, letter_id_t]               # [B, 4]
        preds_idx  = torch.argmax(scores, dim=1).tolist() #antes scores

        for pred_idx, true_letter in zip(preds_idx, true_answers):
            pred_letter = letters[pred_idx]
            per_letter_pred[pred_letter] += 1
            per_letter_true[true_letter]["total"] += 1
            if pred_letter == true_letter:
                correct += 1
                per_letter_true[true_letter]["correct"] += 1
            total += 1

    acc = correct / total
    print(f"\n{desc} -> Accuracy: {acc:.4f}  ({correct}/{total})")
    print(f"Distribución de predicciones: {per_letter_pred}")
    print("Accuracy por letra correcta:")
    for l in letters:
        d = per_letter_true[l]
        if d["total"] > 0:
            print(f"  {l}: {d['correct']}/{d['total']} = {d['correct']/d['total']:.3f}")
    return acc


In [ ]:
@torch.no_grad()
def compute_val_loss(model, dataloader, device="cuda",
                     amp_dtype=torch.bfloat16, desc="Val loss"):
    """Promedio de loss sobre el dataloader de validación."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    for batch in tqdm(dataloader, desc=desc, leave=False):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=amp_dtype):
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
        total_loss += outputs.loss.item()
        n_batches  += 1
    return total_loss / n_batches

In [ ]:
@torch.no_grad()
def generate_examples(model, tokenizer, dataset, n_examples=3,
                      max_new_tokens=20, device="cuda",
                      title="", seed=42):
    """Genera respuestas para n_examples aleatorios del dataset (raw).
    Crea internamente un DataLoader con collate_fn_test para consistencia."""
    model.eval()

    n_examples = min(n_examples, len(dataset))
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(dataset), size=n_examples, replace=False).tolist()

    subset_raw = dataset.select(indices)
    subset_tok = subset_raw.map(
        tokenize_for_testing,
        remove_columns=subset_raw.column_names,
    )
    loader = DataLoader(
        subset_tok, batch_size=n_examples, shuffle=False,
        collate_fn=collate_fn_test,
    )
    batch = next(iter(loader))

    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Con left padding, todos los prompts terminan en la misma columna
    prompt_len      = input_ids.shape[1]
    new_tokens      = out[:, prompt_len:]
    generated_texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f" Ejemplos generados — {title}")
    print(f"{'='*70}")
    dataframe = pd.DataFrame({
        "idx": indices,
        "article": [dataset[int(idx)]["article"] for idx in indices],
        "question": [dataset[int(idx)]["question"] for idx in indices],
        "answer": [dataset[int(idx)]["answer"] for idx in indices],
        "generated": generated_texts,
    })
    return dataframe


## 8. Carga y evaluación del modelo base

Cargamos `Llama-3.2-1B` directamente en `bf16` (o `fp16` en GPUs sin soporte de bf16) para reducir el uso de memoria a aproximadamente la mitad respecto a fp32. La evaluación del modelo **sin entrenar** es la **línea base** sobre la cual mediremos la mejora aportada por el finetuning.

### Referencia: random baseline

Para una tarea de 4 opciones, el accuracy de un clasificador aleatorio es **25%**. Cualquier valor por encima de eso indica que el modelo pretrenado ya extrae alguna señal del prompt en modo *zero-shot*, pero no necesariamente que esté cerca del óptimo para esta tarea.


In [ ]:
import gc
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM libre: {free/1e9:.2f} / {total/1e9:.2f} GB")

In [ ]:
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=AMP_DTYPE,
).to(device)

n_params = sum(p.numel() for p in model_base.parameters())
print(f"Parámetros del modelo: {n_params/1e6:.1f}M")
print(f"Memoria aprox: {n_params * (2 if AMP_DTYPE != torch.float32 else 4) / 1e9:.2f} GB")


In [ ]:
# Evaluación del modelo base sobre test
acc_base = evaluate_accuracy(model_base, test_loader, device=device, desc="Test base")


In [ ]:
# Generación de ejemplos cualitativos del modelo base
generate_examples(model_base, tokenizer, test_ds_raw, n_examples=20,
                  title="Modelo base", max_new_tokens=15)


## 9. Configuración de LoRA


### Iteración de hiperparámetros y justificación

La configuración "de libro" del paper original (`r=8`, `α=16`, solo atención `q_proj`, `v_proj`) dio una mejora modesta sobre la línea base. Se ajustaron los hiperparámetros buscando **más capacidad expresiva** sin perder el carácter *parameter-efficient* del método:

| Hyperparam       | Valor                                       | Justificación                                                                                                                                                                                |
|------------------|---------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `r`              | **32** (antes 8)                            | RACE es una tarea de comprensión lectora real, no una clasificación trivial. Subir el rango aumenta la capacidad del adaptador para representar las direcciones de $\Delta W$ relevantes.    |
| `lora_alpha`     | **64** (antes 16)                           | Se mantiene la razón `α/r = 2` (la misma que la configuración original `r=8, α=16`). Esto preserva la escala efectiva de la actualización y evita tener que retunear el learning rate.        |
| `lora_dropout`   | **0.1** (antes 0.05)                        | Con más parámetros entrenables sobre un dataset filtrado relativamente pequeño, sube el riesgo de overfitting. Un dropout más alto regulariza el adaptador.                                  |
| `target_modules` | `q,k,v,o,gate,up,down` (antes solo `q,v,k,o`) | Extender LoRA a las **proyecciones del MLP** (`gate_proj`, `up_proj`, `down_proj`) fue el cambio que más impactó el accuracy. En Llama el MLP almacena gran parte del *conocimiento factual*; adaptarlo permite ajustar **qué** información se usa para responder, no solo **cómo** se atiende al contexto. |
| `bias`           | `"none"`                                    | No se entrenan los biases — representan una fracción despreciable de los parámetros y rara vez aportan mejoras significativas.                                                                |

### Resumen de la decisión

Pasar de una configuración mínima (`r=8`, solo atención) a una **configuración más expresiva** (`r=32`, atención + MLP, dropout alto) sigue siendo parameter-efficient: aún se entrena un porcentaje pequeño del total del modelo (la celda siguiente lo imprime explícitamente con `print_trainable_parameters()`), pero el adaptador tiene suficiente capacidad para una tarea de QA real.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
)

model = get_peft_model(model_base, lora_config)
model.print_trainable_parameters()


## 10. Entrenamiento

### Configuración

- **Optimizer**: `AdamW` aplicado **únicamente a los parámetros entrenables** (los de LoRA). Los pesos del modelo base no reciben gradiente y, por tanto, tampoco consumen estado del optimizador.
- **Scheduler**: lineal con *warmup* del 5% del total de steps (ver siguiente celda).
- **Mixed precision**: `bf16` preferido; fallback automático a `fp16 + GradScaler` en hardware sin soporte de bf16.
- **Gradient clipping**: norma máxima de `1.0` para estabilizar pasos con gradientes ocasionalmente grandes.
- **Validación por epoch**: se calculan `val_loss` y `val_acc` al final de cada epoch.
- **Checkpointing**: se guarda el **mejor modelo según `val_acc`** (no según `val_loss`), ya que la métrica final del proyecto es el accuracy.
- **Early stopping**: paciencia de 3 epochs sobre `val_acc` con `min_delta = 5e-5`.

### Sobre `GradScaler`

`GradScaler` es necesario únicamente con **fp16**, donde los gradientes pequeños hacen *underflow* a cero debido al rango dinámico limitado del formato (fp16 tiene solo 5 bits de exponente). **bf16** tiene el mismo rango exponencial que fp32 — pierde precisión solo en la mantisa — por lo que no requiere scaler. El código detecta el caso y maneja ambos: `scaler = None` cuando se usa bf16.


### El scheduler: `get_linear_schedule_with_warmup`

Esta función de Hugging Face implementa el siguiente perfil de learning rate a lo largo del entrenamiento:

1. **Fase de warmup** (primeros `num_warmup_steps`): el LR sube **linealmente** desde 0 hasta el valor base configurado (`LR = 2e-4`).
2. **Fase de decay** (steps restantes): el LR baja **linealmente** desde el valor base hasta 0 al final del entrenamiento.

#### Por qué se usa warmup

Al inicio del finetuning, los pesos LoRA están en su inicialización ($B = 0$, $A \sim \mathcal{N}(0, \sigma^2)$), por lo que $\Delta W = BA \approx 0$. Las primeras iteraciones producen gradientes "ruidosos" y aplicar el LR completo desde el step 0 puede desestabilizar el entrenamiento — sobre todo porque Adam aún no ha estimado bien las medias y varianzas de los gradientes. Subir el LR gradualmente le da tiempo al optimizador para acumular estadísticas razonables antes de tomar pasos grandes.

#### Por qué decay lineal

El decay lineal favorece la **convergencia**: a medida que nos acercamos al óptimo, los pasos se hacen más pequeños y se evita oscilar alrededor del mínimo. Es la elección estándar para finetuning corto (pocas epochs) sobre modelos grandes, y es la que el paper original de LoRA recomienda.


In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# fp16 requiere GradScaler; bf16 no
scaler = None if USE_BF16 else GradScaler()

print(f"Total steps:  {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"GradScaler:   {'No (bf16)' if scaler is None else 'Sí (fp16)'}")


### Parámetros de `train_model`

| Parámetro          | Descripción                                                                                                            |
|--------------------|------------------------------------------------------------------------------------------------------------------------|
| `model`            | Modelo PEFT (Llama base + adaptadores LoRA). Solo los pesos LoRA tienen `requires_grad=True`.                          |
| `train_loader`     | DataLoader con **right padding** y labels — apto para el entrenamiento causal.                                          |
| `val_loader`       | DataLoader con **left padding** — usado para calcular `val_acc` mediante lectura de `logits[:, -1, :]`.                 |
| `val_loader_loss`  | DataLoader con **right padding** y labels — usado para calcular la *cross-entropy de validación* en paralelo al accuracy. |
| `optimizer`        | `AdamW` filtrado sobre los parámetros entrenables (los de LoRA).                                                       |
| `scheduler`        | Linear con warmup, definido en la celda anterior.                                                                      |
| `scaler`           | `GradScaler` si se usa fp16; `None` si se usa bf16.                                                                    |
| `epochs`           | Número máximo de epochs (puede terminar antes por early stopping).                                                     |
| `amp_dtype`        | Tipo de mixed precision (`torch.bfloat16` o `torch.float16`).                                                          |
| `grad_clip`        | Norma máxima permitida para el gradiente (clipping global).                                                            |
| `log_every`        | Cada cuántos steps se registra el `train_loss` en el historial.                                                        |
| `patience`         | Número de epochs consecutivos sin mejora antes de detener (early stopping). `None` = sin early stopping.               |
| `min_delta`        | Mejora mínima en `val_acc` requerida para resetear el contador de paciencia.                                           |
| `ckpt_path`        | Ruta donde guardar el mejor checkpoint (formato `torch.save`).                                                         |
| `extra_ckpt_info`  | Dict adicional que se anexa al checkpoint (p. ej. el `lora_config`, hiperparámetros, etc.).                            |

La función devuelve un dict con `history` (curvas de loss y accuracy por step/epoch), `best_val_acc`, `last_epoch`, `stopped_early` y `ckpt_path`.


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,            # left padding, test-style (para accuracy)
    val_loader_loss,       # right padding, train-style (para loss)
    optimizer,
    scheduler,
    scaler=None,           # None para bf16, GradScaler() para fp16
    *,
    epochs=8,
    device="cuda",
    amp_dtype=torch.bfloat16,
    grad_clip=1.0,
    log_every=50,
    patience=3,            # epochs sin mejora antes de parar; None = no early stop
    min_delta=0.00005,       # mejora mínima en val_acc para resetear paciencia
    ckpt_path=None,
    extra_ckpt_info=None,  # dict adicional a guardar en el checkpoint
):
    """Entrena con validación por epoch (loss + accuracy) y early stopping
    basado en val_acc.

    Returns:
        dict con keys: history, best_val_acc, last_epoch, stopped_early, ckpt_path
    """
    history = {
        "step":     [], "train_loss": [],
        "val_step": [], "val_loss":   [], "val_acc": [],
    }
    best_val_acc      = -1.0
    patience_counter  = 0
    stopped_early     = False
    last_epoch        = 0
    global_step       = 0

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(epochs):
        last_epoch = epoch + 1
        model.train()
        epoch_losses = []
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        # ------- Training loop -------
        for batch in pbar:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=amp_dtype):
                outputs = model(input_ids=input_ids,
                                attention_mask=attention_mask,
                                labels=labels)
                loss = outputs.loss

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable_params, grad_clip)
                optimizer.step()

            scheduler.step()
            global_step += 1
            epoch_losses.append(loss.item())

            if global_step % log_every == 0:
                recent = epoch_losses[-log_every:]
                avg_loss = sum(recent) / len(recent)
                history["step"].append(global_step)
                history["train_loss"].append(avg_loss)
                pbar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
                })

        # ------- Validación al final de cada epoch -------
        val_loss = compute_val_loss(
            model, val_loader_loss, device=device,
            amp_dtype=amp_dtype,
            desc=f"Val loss epoch {epoch+1}",
        )
        val_acc = evaluate_accuracy(
            model, val_loader, device=device,
            desc=f"Val acc epoch {epoch+1}",
        )

        train_loss_epoch = sum(epoch_losses) / len(epoch_losses)
        history["val_step"].append(global_step)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"\nEpoch {epoch+1}: "
              f"train_loss={train_loss_epoch:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f}")

        # ------- Checkpoint + early stopping -------
        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            patience_counter = 0
            checkpoint = {
                "model_state_dict": get_peft_model_state_dict(model),
                "epoch":            epoch + 1,
                "val_acc":          val_acc,
                "val_loss":         val_loss,
                "global_step":      global_step,
            }
            if extra_ckpt_info:
                checkpoint.update(extra_ckpt_info)
            if ckpt_path is not None:
                torch.save(checkpoint, ckpt_path)
                print(f"✓ Nuevo mejor val_acc: {val_acc:.4f} → checkpoint guardado")
        else:
            patience_counter += 1
            patience_str = f"{patience_counter}/{patience}" if patience else f"{patience_counter}/∞"
            print(f"  Sin mejora significativa ({patience_str} epochs)")
            if patience is not None and patience_counter >= patience:
                print(f"\n⚠ Early stopping en epoch {epoch+1}: "
                      f"{patience} epochs sin mejora >{min_delta:.3f}")
                stopped_early = True
                break

    # ------- Resumen final -------
    print(f"\n{'='*50}")
    print(f"Mejor val_acc: {best_val_acc:.4f}")
    if stopped_early:
        print(f"Detenido por early stopping en epoch {last_epoch}/{epochs}")
    else:
        print(f"Completó las {epochs} epochs")

    return {
        "history":       history,
        "best_val_acc":  best_val_acc,
        "last_epoch":    last_epoch,
        "stopped_early": stopped_early,
        "ckpt_path":     ckpt_path,
    }

In [ ]:
result = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_loader_loss=val_loader_loss,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    epochs=EPOCHS,
    device=device,
    amp_dtype=AMP_DTYPE,
    grad_clip=GRAD_CLIP,
    log_every=50,
    patience=3,
    min_delta=0.00005,
    ckpt_path=os.path.join(SAVE_PATH, "best_model_lora.pt"),
    extra_ckpt_info={"lora_config": lora_config.to_dict()},
)

# Desempaquetar para usar después
history        = result["history"]
best_val_acc   = result["best_val_acc"]
best_ckpt_path = result["ckpt_path"]

## 11. Evaluación final del modelo finetuneado

Para garantizar una evaluación **justa y reproducible**:

1. Liberamos memoria GPU eliminando las referencias al modelo base y al PEFT model que quedaron en RAM tras el entrenamiento.
2. **Recargamos** `Llama-3.2-1B` desde Hugging Face desde cero y le aplicamos el mismo `lora_config`.
3. Cargamos el **mejor checkpoint** (el del epoch con mayor `val_acc`) usando `set_peft_model_state_dict`.
4. Evaluamos sobre el split de **test** filtrado (artículos con menos de 800 caracteres).

Este protocolo asegura que el accuracy reportado corresponde al checkpoint **óptimo según validación**, no al último del entrenamiento (que podría haber empezado a sobreajustar). También se generan ejemplos cualitativos sobre el conjunto de test para comparar visualmente con los del modelo base.


In [ ]:
try:
    del model_base
except NameError:
    pass
try:
    del model
except NameError:
    pass

free_memory()

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
)

In [ ]:
model_base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
model = get_peft_model(model_base, lora_config)

In [ ]:
try:
    if best_ckpt_path is None:
        best_ckpt_path=os.path.join('.', "best_model_lora.pt")
except NameError:
    best_ckpt_path=os.path.join('.', "best_model_lora.pt")

In [ ]:
from peft import set_peft_model_state_dict

ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
set_peft_model_state_dict(model, ckpt["model_state_dict"])
model = model.to(device)
print(f"Cargado checkpoint: epoch {ckpt['epoch']}, val_acc {ckpt['val_acc']:.4f}")

acc_ft = evaluate_accuracy(model, test_loader, device=device, desc="Test finetuneado")


In [ ]:
generate_examples(model, tokenizer, test_ds_raw, n_examples=20,
                  title="Modelo finetuneado", max_new_tokens=15)


## 12. Análisis y comparación de resultados

La siguiente celda produce **tres gráficos** para sintetizar el experimento:

1. **Curvas de pérdida** (`train_loss` vs. `val_loss`): permite detectar overfitting. Si `train_loss` sigue bajando mientras `val_loss` empieza a subir, el modelo está memorizando el set de entrenamiento.
2. **Val accuracy por epoch**: muestra la evolución del accuracy de validación a lo largo del entrenamiento, con el mejor punto marcado (el que corresponde al checkpoint guardado).
3. **Comparativa final**: barras del accuracy sobre **test** del modelo base vs. el finetuneado, con la mejora absoluta en puntos porcentuales.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["step"], history["train_loss"],
             color="steelblue", alpha=0.7, label="Train loss")
axes[0].plot(history["val_step"], history["val_loss"], "o-",
             color="darkorange", markersize=8, linewidth=2, label="Val loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Pérdida: train vs val")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history["val_step"], history["val_acc"], "o-",
             color="seagreen", markersize=8, linewidth=2)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Val accuracy")
axes[1].set_title("Validation accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

best_idx = int(np.argmax(history["val_acc"]))
axes[1].annotate(f"Best: {history['val_acc'][best_idx]:.3f}",
                 xy=(history["val_step"][best_idx], history["val_acc"][best_idx]),
                 xytext=(10, -15), textcoords="offset points",
                 fontsize=9, color="seagreen", fontweight="bold")

bars = axes[2].bar(["Base", "Finetuneado"], [acc_base, acc_ft],
                    color=["lightcoral", "seagreen"], edgecolor="black")
axes[2].set_ylabel("Test accuracy")
axes[2].set_title(f"Mejora: +{(acc_ft - acc_base)*100:.2f} puntos")
axes[2].set_ylim(0, max(acc_base, acc_ft) * 1.25)
for bar, v in zip(bars, [acc_base, acc_ft]):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                 f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "comparacion.png"), dpi=100, bbox_inches="tight")
plt.show()

print(f"\n=== Resultados finales ===")
print(f"  Test accuracy base:        {acc_base:.4f}")
print(f"  Test accuracy finetuneado: {acc_ft:.4f}")
print(f"  Mejora absoluta:           +{(acc_ft - acc_base)*100:.2f} pp")
print(f"  Mejora relativa:           +{(acc_ft - acc_base)/acc_base*100:.1f}%")


## 13. Conclusiones

### Resultados cuantitativos

- El **modelo base** `Llama-3.2-1B`, evaluado sin entrenar (*zero-shot*), supera el random baseline del 25%, lo que indica que el modelo pretrenado ya posee comprensión lectora suficiente para extraer alguna señal del prompt; sin embargo, está lejos del óptimo para esta tarea específica.
- Tras el finetuning con LoRA, el accuracy en test mejora **sustancialmente** respecto a la línea base, confirmando la hipótesis del paper original: un adaptador de bajo rango es suficiente para especializar un LLM en una tarea de QA de selección múltiple sin tocar la inmensa mayoría de sus parámetros.

### Por qué la configuración de LoRA elegida funcionó

La configuración "de libro" del paper (`r=8`, `α=16`, solo `q_proj` y `v_proj`) es óptima para tareas donde la señal de la nueva tarea reside principalmente en las relaciones atencionales (clasificación, NLI, MNLI, etc.). Para RACE — una tarea de comprensión lectora real — los siguientes cambios fueron decisivos:

1. **Subir `r` de 8 a 32**: dio al adaptador capacidad suficiente para representar las direcciones del cambio de pesos asociadas a comprensión lectora, no solo a reconocer un patrón superficial.
2. **Adaptar también el MLP** (`gate_proj`, `up_proj`, `down_proj`): en Llama el MLP almacena gran parte del conocimiento factual y semántico. Adaptarlo permitió ajustar **qué** información se usa para responder, no solo **cómo** se atiende al contexto. Este fue el cambio que más impactó el accuracy en nuestras iteraciones.
3. **Mantener `α/r = 2`**: al subir `r`, subimos `α` proporcionalmente para no tener que retunear el learning rate.
4. **Subir el dropout a 0.1**: con más parámetros entrenables sobre un dataset filtrado relativamente pequeño, la regularización por dropout fue necesaria para evitar overfitting (observable en la curva de `val_loss`).

### Decisiones técnicas que importaron

- **Tokenización con espacio inicial (`" A"`)**: garantiza que cada opción sea **un único token**, permitiendo medir $P(s \mid c)$ con un solo forward pass — fiel a la fórmula del enunciado y mucho más eficiente que generar texto.
- **Enmascarado del contexto con `-100`**: el modelo solo es penalizado por predecir la respuesta, no por reproducir el artículo. Sin esto, la señal de entrenamiento se diluye y el modelo aprende a copiar en lugar de a responder.
- **Padding asimétrico** (right en training, left en evaluación): permite que `logits[:, -1, :]` apunte uniformemente al último token real para todo el batch en evaluación, evitando errores silenciosos en el cómputo del accuracy.
- **`bf16` sobre `fp16`**: simplifica el código (sin `GradScaler`) y es completamente estable en GPUs Ampere y posteriores.
- **Checkpointing por `val_acc`** (no por `val_loss`): alineado con la métrica final del proyecto.

### Análisis cualitativo

Las generaciones del modelo finetuneado muestran un patrón muy distinto al del base: el modelo base tiende a generar texto adicional (parafraseando opciones, repitiendo la pregunta, intentando explicar) en lugar de comprometerse con una letra, mientras que el finetuneado emite **la letra correcta directamente**, seguida del token EOS. Esto es consistente con haber aprendido el formato exacto de respuesta `" <letter> EOS"` durante el entrenamiento.

La **distribución de predicciones por letra** que imprime `evaluate_accuracy` también es informativa: en el modelo base se observa cierto sesgo hacia alguna opción concreta, mientras que en el finetuneado la distribución se acerca más a la distribución real de la clase correcta en el set de test.

### Limitaciones

- Solo se trabajó con el subconjunto filtrado (`len(article) < 800`). El comportamiento sobre artículos más largos **no está garantizado** y probablemente requeriría reentrenamiento con `MAX_LENGTH` mayor.
- El experimento se ejecutó con **un solo seed**; no se reporta la varianza entre seeds, lo que sería deseable para distinguir mejoras reales de ruido.
- No se hizo una búsqueda exhaustiva del espacio de hiperparámetros de LoRA. La configuración final fue resultado de unas pocas iteraciones manuales, no de un grid o random search sistemático.
- El modelo aprendió el formato específico de respuesta (`" A"`/`" B"`/`" C"`/`" D"`). Si se cambiara el prompt para esperar `"Option A"` o la frase completa de la opción, el modelo finetuneado probablemente no transferiría bien.


### Desconexión automática del runtime

La siguiente celda libera el runtime de Colab automáticamente al final del notebook para ahorrar recursos de GPU. **Comentar la celda si se desea inspeccionar el estado tras la ejecución** (variables, modelo cargado en GPU, etc.).


In [ ]:
from google.colab import runtime
runtime.unassign()